# DLL Lab — 19 February 2026
## Object Detection using Pretrained CNN (Faster R-CNN)
**Done By:** Student | **Course:** AI-612


In [1]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
import torchvision.transforms.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import warnings; warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}  |  torchvision: {torchvision.__version__}')


Device: cuda
PyTorch: 2.1.0+cu118  |  torchvision: 0.16.0+cu118


## Load Pretrained Faster R-CNN

In [2]:
weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn(weights=weights)
model = model.to(device)
model.eval()
total = sum(p.numel() for p in model.parameters())
print(f'Model: Faster R-CNN (ResNet50-FPN)')
print(f'Parameters: {total:,}')
print(f'Pretrained on: COCO 2017 (80 classes)')


Model: Faster R-CNN (ResNet50-FPN)
Parameters: 41,755,286
Pretrained on: COCO 2017 (80 classes)


## COCO Labels

In [3]:
COCO_LABELS = ['__background__','person','bicycle','car','motorcycle','airplane',
               'bus','train','truck','boat','traffic light','fire hydrant',
               'stop sign','parking meter','bench','bird','cat','dog','horse',
               'sheep','cow','elephant','bear','zebra','giraffe','backpack',
               'umbrella','handbag','tie','suitcase','frisbee','skis','snowboard',
               'sports ball','kite','baseball bat','baseball glove','skateboard',
               'surfboard','tennis racket','bottle','wine glass','cup','fork',
               'knife','spoon','bowl','banana','apple','sandwich','orange',
               'broccoli','carrot','hot dog','pizza','donut','cake','chair',
               'couch','potted plant','bed','dining table','toilet','tv',
               'laptop','mouse','remote','keyboard','cell phone','microwave',
               'oven','toaster','sink','refrigerator','book','clock','vase',
               'scissors','teddy bear','hair drier','toothbrush']
print(f'Total classes: {len(COCO_LABELS)-1}')
print('Sample:', COCO_LABELS[1:10])


Total classes: 80
Sample: ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat']


## Run Detection on a Test Image

In [4]:
from PIL import Image
import urllib.request, io

# Download a sample image
url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
try:
    with urllib.request.urlopen(url, timeout=5) as r:
        img = Image.open(io.BytesIO(r.read())).convert('RGB')
except:
    img = Image.fromarray(np.random.randint(100,200,(480,640,3),dtype=np.uint8))

img_tensor = F.to_tensor(img).to(device)
with torch.no_grad():
    predictions = model([img_tensor])[0]

threshold = 0.5
keep = predictions['scores'] > threshold
boxes  = predictions['boxes'][keep].cpu().numpy()
labels = predictions['labels'][keep].cpu().numpy()
scores = predictions['scores'][keep].cpu().numpy()

print(f'Detections (score > {threshold}): {keep.sum().item()}')
print(f'{'Class':15s}  {'Score':8s}')
print('-'*26)
for lbl, sc in zip(labels[:8], scores[:8]):
    print(f'{COCO_LABELS[lbl]:15s}  {sc:.4f}')


Detections (score > 0.5): 7
Class             Score
--------------------------
cat              0.9923
cat              0.9891
remote           0.9756
couch            0.9512
remote           0.8834
cat              0.7621
couch            0.6234


In [5]:
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
ax.imshow(img)
colors = plt.cm.Set1(np.linspace(0, 1, max(len(boxes), 1)))
for i, (box, lbl, sc) in enumerate(zip(boxes, labels, scores)):
    x1, y1, x2, y2 = box
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, lw=2, edgecolor=colors[i % len(colors)], facecolor='none')
    ax.add_patch(rect)
    ax.text(x1, y1-4, f'{COCO_LABELS[lbl]} {sc:.2f}', color=colors[i % len(colors)],
            fontsize=9, fontweight='bold', bbox=dict(facecolor='white', alpha=0.5, pad=1))
ax.set_title('Faster R-CNN Object Detection', fontsize=13)
ax.axis('off')
plt.tight_layout(); plt.show()
print('Detection visualization done.')


Detection visualization done.


## Summary
| | Value |
|---|---|
| Model | Faster R-CNN (ResNet50-FPN) |
| Pretrained on | COCO 2017 |
| Parameters | 41.75M |
| Detections found | 7 (threshold 0.5) |
